## Setup

In [ ]:
import pandas as pd
from pathlib import Path

CORPUS_PATH = Path("../collection/data/corpus_deduped.parquet")

corpus = pd.read_parquet(CORPUS_PATH, engine="pyarrow")
print(f"Loaded full corpus: {len(corpus)} comments")
print(corpus.columns.tolist())

## Random Seed Sampling
Randomly permutes every comment in the full corpus once, under a fixed seed, and saves the resulting order. Downstream filtering/stratification cells draw from this file rather than re-randomizing, so the same seed always produces the same base ordering.

In [ ]:
SEED = 0

seeds_dir = Path("seeds")
seeds_dir.mkdir(parents=True, exist_ok=True)

shuffled = corpus.sample(frac=1, random_state=SEED).reset_index(drop=True)

seed_path = seeds_dir / f"random_seed_{SEED}.parquet"
shuffled.to_parquet(seed_path, engine="pyarrow", index=False)
print(f"Saved {len(shuffled)} comments in random order (seed={SEED}) to {seed_path}")


## Filter and Stratify
Filters the randomly ordered corpus by text length (character and word count), then draws a stratified sample, adjustable per stratum by `N_PER_STRATUM`. Because the source is already randomized, taking the first `N` rows per stratum group is itself a random draw within that stratum, no re-sampling needed.

In [ ]:
# Filtering and Stratification Parameters
MIN_CHARS = None      # e.g. 20
MAX_CHARS = None      # e.g. 2000
MIN_WORDS = None      # e.g. 3
MAX_WORDS = None      # e.g. 300

STRATIFY_BY = ["platform", "stratum"]   # columns to stratify on
N_PER_STRATUM = 50                      # target N per stratum combination

TEXT_COL = "_source.text"

df = shuffled.copy()

df["_char_len"] = df[TEXT_COL].fillna("").str.len()
df["_word_len"] = df[TEXT_COL].fillna("").str.split().str.len()

if MIN_CHARS is not None:
    df = df[df["_char_len"] >= MIN_CHARS]
if MAX_CHARS is not None:
    df = df[df["_char_len"] <= MAX_CHARS]
if MIN_WORDS is not None:
    df = df[df["_word_len"] >= MIN_WORDS]
if MAX_WORDS is not None:
    df = df[df["_word_len"] <= MAX_WORDS]

print(f"After length filtering: {len(df)} comments")

sampled = (
    df.groupby(STRATIFY_BY, group_keys=False)
    .apply(lambda g: g.head(N_PER_STRATUM))
    .reset_index(drop=True)
)

print(f"After stratified sampling ({N_PER_STRATUM} per {STRATIFY_BY}): {len(sampled)} comments")
print(sampled.groupby(STRATIFY_BY).size().to_string())


## Save Sample
Saves the filtered, stratified sample under a filename built directly from the parameters used to produce it, so the file's name is a full record of how it was generated.

In [ ]:
samples_dir = Path("samples")
samples_dir.mkdir(parents=True, exist_ok=True)

def fmt(val):
    return "none" if val is None else str(val)

filename_parts = [
    f"seed{SEED}",
    f"minchar{fmt(MIN_CHARS)}",
    f"maxchar{fmt(MAX_CHARS)}",
    f"minword{fmt(MIN_WORDS)}",
    f"maxword{fmt(MAX_WORDS)}",
    f"n{N_PER_STRATUM}",
    "by-" + "-".join(STRATIFY_BY),
]
sample_filename = "_".join(filename_parts) + ".parquet"
sample_path = samples_dir / sample_filename

sampled.to_parquet(sample_path, engine="pyarrow", index=False)
print(f"Saved {len(sampled)} comments to {sample_path}")
